<a href="https://colab.research.google.com/github/MBKNgcobo/GF.github.io/blob/main/sequential_task(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Forecast California Housing Prices

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from time import time

# Dataset
from sklearn.datasets import fetch_california_housing

In [ ]:
# Set random seed
SEED = 50
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [ ]:
# Load the dataset
data = pd.read_csv('/content/sample_data/housing.csv')

# Assuming 'median_house_value' is the target column based on the California housing dataset
target_column = 'median_house_value'
features = data.drop(columns=[target_column])
targets = data[target_column].values.astype(float).reshape(-1, 1)

**Questions**

* What is a `Bunch`?
* What is the default type for `data`?
* What is the default type for `target`?

**Answers**

1.A `Bunch` is a small container object used by scikit-learn that behaves like a dictionary but also exposes keys as attributes (so you can use `bunch['data']` **or** `bunch.data`).


2.By default `data` is returned as a NumPy `ndarray` (for the California housing loader: `data` has shape `[20640, 8]`).


3.By default `target` is returned as a NumPy array of shape `(20640,)` (each value is the median house value for a district).


In [ ]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder

# Load data
df = pd.read_csv('/content/sample_data/housing.csv')

# Separate target
y = df['median_house_value']

# Separate features
X = df.drop(columns=['median_house_value'])

# Identify categorical and numerical columns
categorical_cols = ['ocean_proximity']
numerical_cols = X.columns.drop(categorical_cols)

# Scale numerical features
num_scaler = MinMaxScaler()
X_num_scaled = num_scaler.fit_transform(X[numerical_cols])

# One-hot encode categorical feature
encoder = OneHotEncoder(sparse_output=False)
X_cat_encoded = encoder.fit_transform(X[categorical_cols])

# Combine scaled numerical + encoded categorical features
import numpy as np
X_scaled = np.hstack([X_num_scaled, X_cat_encoded])

# Scale target
y_scaler = MinMaxScaler()
y_scaled = y_scaler.fit_transform(y.values.reshape(-1, 1))

# Check shapes
print(X_scaled.shape)
print(y_scaled.shape)


(20640, 13)
(20640, 1)


In [ ]:
# Normalization cell -> produces features_normalized and targets_normalized
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder


if 'median_house_value' not in df.columns:
    raise KeyError("Expected target column 'median_house_value' in df")

# Separate features / target
X_df = df.drop(columns=['median_house_value'])
y = df['median_house_value'].values.reshape(-1, 1)   # 2D for scaler

# Detect numeric and categorical columns automatically
numeric_cols = X_df.select_dtypes(include=['number']).columns.tolist()
categorical_cols = X_df.select_dtypes(include=['object', 'category']).columns.tolist()

# Pipelines
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', MinMaxScaler())
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# ColumnTransformer to apply transformations
transformers = []
if numeric_cols:
    transformers.append(('num', numeric_pipeline, numeric_cols))
if categorical_cols:
    transformers.append(('cat', categorical_pipeline, categorical_cols))

preprocessor = ColumnTransformer(transformers, remainder='drop')

# Fit & transform features
X_transformed = preprocessor.fit_transform(X_df)

# Try to get feature names (optional). If not available, skip.
try:
    feature_names = preprocessor.get_feature_names_out()
    # if you want a DataFrame (optional), uncomment the next line:
    # X_transformed_df = pd.DataFrame(X_transformed, columns=feature_names)
except Exception:
    feature_names = None

# Scale target (median_house_value) to [0,1]
y_scaler = MinMaxScaler()
y_scaled = y_scaler.fit_transform(y).ravel()   # 1D array

# Final variable names required for your sequence function
features_normalized = np.asarray(X_transformed)      # shape: (n_samples, n_features)
targets_normalized = np.asarray(y_scaled)            # shape: (n_samples,)

# Diagnostics
print("features_normalized shape:", features_normalized.shape)
print("targets_normalized shape:", targets_normalized.shape)
if feature_names is not None:
    print("Number of feature columns (named):", len(feature_names))
    print("First 8 feature names:", feature_names[:8])


features_normalized shape: (20640, 13)
targets_normalized shape: (20640,)
Number of feature columns (named): 13
First 8 feature names: ['num__longitude' 'num__latitude' 'num__housing_median_age'
 'num__total_rooms' 'num__total_bedrooms' 'num__population'
 'num__households' 'num__median_income']


In [ ]:
def create_sequences(data, targets, seq_length):
   X, y = [], []
   for i in range(len(data) - seq_length):
       X.append(data[i:i+seq_length])
       y.append(targets[i+seq_length])
   return np.array(X), np.array(y)


seq_length = 20
X, y = create_sequences(features_normalized, targets_normalized, seq_length)

# Data cleaning and preprocessing

In [ ]:
import torch
from sklearn.model_selection import train_test_split

# --- Train / Test split  ---
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.33,
    shuffle=False
)

# --- Convert NumPy arrays to PyTorch tensors ---
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor  = torch.tensor(X_test, dtype=torch.float32)

y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
y_test_tensor  = torch.tensor(y_test, dtype=torch.float32)

# --- Diagnostics ---
print("Training set shapes:")
print("X_train_tensor:", X_train_tensor.shape)
print("y_train_tensor:", y_train_tensor.shape)

print("\nTesting set shapes:")
print("X_test_tensor:", X_test_tensor.shape)
print("y_test_tensor:", y_test_tensor.shape)


Training set shapes:
X_train_tensor: torch.Size([13815, 20, 13])
y_train_tensor: torch.Size([13815])

Testing set shapes:
X_test_tensor: torch.Size([6805, 20, 13])
y_test_tensor: torch.Size([6805])


In [ ]:
# Check data types
df.info()

# Check missing values
df.isnull().sum()

# Check duplicate rows
print("Duplicate rows:", df.duplicated().sum())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   longitude           20640 non-null  float64
 1   latitude            20640 non-null  float64
 2   housing_median_age  20640 non-null  float64
 3   total_rooms         20640 non-null  float64
 4   total_bedrooms      20433 non-null  float64
 5   population          20640 non-null  float64
 6   households          20640 non-null  float64
 7   median_income       20640 non-null  float64
 8   median_house_value  20640 non-null  float64
 9   ocean_proximity     20640 non-null  object 
dtypes: float64(9), object(1)
memory usage: 1.6+ MB
Duplicate rows: 0


In [ ]:
# Remove duplicate rows
df = df.drop_duplicates()

# Convert categorical columns to category dtype
for col in df.select_dtypes(include=['object']).columns:
    df[col] = df[col].astype('category')

# Handle missing values
# Numeric: median
numeric_cols = df.select_dtypes(include=['number']).columns
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())

# Categorical: mode
categorical_cols = df.select_dtypes(include=['category']).columns
for col in categorical_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

print("Cleaned dataset shape:", df.shape)


Cleaned dataset shape: (20640, 10)


In [ ]:
# Separate features and target
X_df = df.drop(columns=['median_house_value'])
y_df = df['median_house_value']

print("Features shape:", X_df.shape)
print("Target shape:", y_df.shape)


Features shape: (20640, 9)
Target shape: (20640,)


In [ ]:
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

# Identify columns
numeric_cols = X_df.select_dtypes(include=['number']).columns
categorical_cols = X_df.select_dtypes(include=['category']).columns

# Pipelines
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', MinMaxScaler())
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Combine preprocessing
preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, numeric_cols),
    ('cat', categorical_pipeline, categorical_cols)
])

# Transform features
features_normalized = preprocessor.fit_transform(X_df)

# Scale target
target_scaler = MinMaxScaler()
targets_normalized = target_scaler.fit_transform(
    y_df.values.reshape(-1, 1)
).ravel()

# Verify
print("features_normalized shape:", features_normalized.shape)
print("targets_normalized shape:", targets_normalized.shape)


features_normalized shape: (20640, 13)
targets_normalized shape: (20640,)


# LSTM model

In [ ]:
class LSTMRegressor(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=2, output_size=1, dropout=0.0):
        super(LSTMRegressor, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        # batch_first=True so input shape is (batch, seq, feature)
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, dropout=dropout)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        # x: (batch, seq_len, input_size)
        out, (hn, cn) = self.lstm(x)  # out: (batch, seq_len, hidden_size)
        # take the last timestep's output
        last = out[:, -1, :]          # (batch, hidden_size)
        out = self.fc(last)           # (batch, output_size)
        return out

In [ ]:
input_size = X_train_tensor.shape[2]
hidden_size = 128          # changeable
num_layers = 2             # changeable
output_size = 1
dropout = 0.0

learning_rate = 1e-3       # changeable
weight_decay = 1e-5        # L2 regularization via optimizer's weight_decay
batch_size = 64
num_epochs = 40            # changeable
print(f"Model will use input_size={input_size}, hidden_size={hidden_size}, num_layers={num_layers}")

Model will use input_size=13, hidden_size=128, num_layers=2


In [ ]:
from torch.utils.data import TensorDataset, DataLoader

model = LSTMRegressor(input_size, hidden_size, num_layers, output_size, dropout).to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

# --- 8) DataLoader ---
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset  = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=False)
test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
for epoch in range(1, num_epochs + 1):
    model.train()
    epoch_losses = []
    for xb, yb in train_loader:
        optimizer.zero_grad()
        out = model(xb)
        loss = criterion(out, yb)
        loss.backward()
        optimizer.step()
        epoch_losses.append(loss.item())
    epoch_loss = float(np.mean(epoch_losses)) if epoch_losses else 0.0
    print(f"Epoch {epoch:3d}/{num_epochs} — train_loss: {epoch_loss:.6f}")

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/loss.py:634: UserWarning: Using a target size (torch.Size([64])) that is different to the input size (torch.Size([64, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/loss.py:634: UserWarning: Using a target size (torch.Size([64])) that is different to the input size (torch.Size([64, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/loss.py:634: UserWarning: Using a target size (torch.Size([64])) that is different to the input size (torch.Size([64, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input,

Epoch   1/40 — train_loss: 0.057394
Epoch   2/40 — train_loss: 0.054383
Epoch   3/40 — train_loss: 0.054652
Epoch   4/40 — train_loss: 0.054429
Epoch   5/40 — train_loss: 0.054339
Epoch   6/40 — train_loss: 0.054465
Epoch   7/40 — train_loss: 0.054318
Epoch   8/40 — train_loss: 0.054376
Epoch   9/40 — train_loss: 0.054434
Epoch  10/40 — train_loss: 0.054306
Epoch  11/40 — train_loss: 0.054283
Epoch  12/40 — train_loss: 0.054276
Epoch  13/40 — train_loss: 0.054229
Epoch  14/40 — train_loss: 0.054355
Epoch  15/40 — train_loss: 0.054173
Epoch  16/40 — train_loss: 0.054241
Epoch  17/40 — train_loss: 0.054236
Epoch  18/40 — train_loss: 0.054258
Epoch  19/40 — train_loss: 0.054275
Epoch  20/40 — train_loss: 0.054229
Epoch  21/40 — train_loss: 0.054319
Epoch  22/40 — train_loss: 0.054225
Epoch  23/40 — train_loss: 0.054192
Epoch  24/40 — train_loss: 0.054194
Epoch  25/40 — train_loss: 0.054353
Epoch  26/40 — train_loss: 0.054299
Epoch  27/40 — train_loss: 0.054254
Epoch  28/40 — train_loss: 0

In [ ]:
# --- 10) Predictions on test set ---
model.eval()
preds = []
with torch.no_grad():
    for xb, _ in test_loader:
        out = model(xb)
        preds.append(out.cpu().numpy())
preds = np.vstack(preds)            # shape (n_test, 1)
y_test_np = y_test_tensor.cpu().numpy()  # shape (n_test, 1)

In [ ]:
# --- 11) Invert scaling of predictions and true targets to original range ---
# Try to find the scaler used for target in previous cells (y_scaler or target_scaler)
scaler = None
if 'y_scaler' in globals():
    scaler = globals()['y_scaler']
elif 'target_scaler' in globals():
    scaler = globals()['target_scaler']
elif 'y_scaled' in globals() and 'df' in globals():
    # try to refit a MinMaxScaler on original df median_house_value
    from sklearn.preprocessing import MinMaxScaler
    scaler = MinMaxScaler()
    scaler.fit(df['median_house_value'].values.reshape(-1, 1))
    print("Refit a MinMaxScaler on df['median_house_value'] for inverse transform.")
else:
    # As a last resort, try to infer min/max from targets_normalized and original df if present
    try:
        from sklearn.preprocessing import MinMaxScaler
        scaler = MinMaxScaler()
        if 'df' in globals():
            scaler.fit(df['median_house_value'].values.reshape(-1, 1))
            print("Refit a MinMaxScaler on df['median_house_value'] for inverse transform (fallback).")
        else:
            raise RuntimeError("Unable to find target scaler and df not available to refit.")
    except Exception as e:
        raise RuntimeError("Can't invert scaling: no target scaler (y_scaler/target_scaler) found and df not available.") from e

# Inverse transform (scaler expects 2D arrays)
preds_orig = scaler.inverse_transform(preds)
y_test_orig = scaler.inverse_transform(y_test_np.reshape(-1, 1))

In [ ]:
# --- 12) Compute and print MSE in original units ---
mse = mean_squared_error(y_test_orig, preds_orig)
print(f"\nTest MSE (original target units): {mse:.4f}")


Test MSE (original target units): 14812807168.0000


In [ ]:
# ---- Experiment configurations ----

input_size = X_train_tensor.shape[2]
base_epochs = 30
experiments = []

# No regularization: LSTM baseline
experiments.append({
    'model_name': 'LSTM_no_reg',
    'model_class': LSTMRegressor,
    'config': {
        'input_size': input_size,
        'hidden_size': 64,
        'num_layers': 1,
        'dropout': 0.0,
        'lr': 1e-3,
        'weight_decay': 0.0,   # NO L2
        'batch_size': 64,
        'epochs': base_epochs
    }
})

In [ ]:
# LSTM configs (with L2)
experiments.append({
    'model_name': 'LSTM_reg_configA',
    'model_class': LSTMRegressor,
    'config': {
        'input_size': input_size,
        'hidden_size': 128,
        'num_layers': 2,
        'dropout': 0.2,
        'lr': 1e-3,
        'weight_decay': 1e-5,    # L2 regularization
        'batch_size': 64,
        'epochs': base_epochs
    }
})

experiments.append({
    'model_name': 'LSTM_reg_configB',
    'model_class': LSTMRegressor,
    'config': {
        'input_size': input_size,
        'hidden_size': 256,
        'num_layers': 2,
        'dropout': 0.3,
        'lr': 5e-4,
        'weight_decay': 1e-4,    # stronger L2
        'batch_size': 64,
        'epochs': base_epochs
    }
})

# GRU model

In [ ]:
class GRURegressor(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=1, output_size=1, dropout=0.0):
        super().__init__()
        self.gru = nn.GRU(input_size, hidden_size, num_layers, batch_first=True, dropout=dropout)
        self.fc = nn.Linear(hidden_size, output_size)
    def forward(self, x):
        out, _ = self.gru(x)
        last = out[:, -1, :]
        return self.fc(last)

In [ ]:
# ---- Training / evaluation helper ----
def train_and_eval(model_class, config, X_train, y_train, X_test, y_test, scaler=None):
    """
    config: dict with keys:
      - input_size, hidden_size, num_layers, dropout, lr, weight_decay, batch_size, epochs
    """
    input_size = config['input_size']
    model = model_class(input_size, config['hidden_size'], config['num_layers'], output_size=1, dropout=config.get('dropout',0.0)).to(device)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=config['lr'], weight_decay=config['weight_decay'])
    train_ds = TensorDataset(X_train, y_train)
    test_ds  = TensorDataset(X_test, y_test)
    train_loader = DataLoader(train_ds, batch_size=config['batch_size'], shuffle=True)
    test_loader  = DataLoader(test_ds, batch_size=config['batch_size'], shuffle=False)

    # Train
    for epoch in range(1, config['epochs']+1):
        model.train()
        losses = []
        for xb, yb in train_loader:
            optimizer.zero_grad()
            out = model(xb)
            loss = criterion(out, yb)
            loss.backward()
            optimizer.step()
            losses.append(loss.item())
        # Optionally could print per-epoch; but we keep it quiet for faster results
    # Evaluate on test set
    model.eval()
    preds = []
    ys = []
    with torch.no_grad():
        for xb, yb in test_loader:
            out = model(xb).cpu().numpy()
            preds.append(out)
            ys.append(yb.cpu().numpy().reshape(-1, 1)) # Reshape yb to be 2D for stacking
    preds = np.vstack(preds)  # (N_test,1)
    ys = np.vstack(ys)
    # Inverse scale if scaler exists
    if scaler is not None:
        preds_orig = scaler.inverse_transform(preds)
        ys_orig = scaler.inverse_transform(ys)
        mse_orig = mean_squared_error(ys_orig, preds_orig)
    else:
        mse_orig = mean_squared_error(ys, preds)  # MSE on normalized scale
    return mse_orig, model, preds, ys

In [ ]:
# No regularization: GRU baseline
experiments.append({
    'model_name': 'GRU_no_reg',
    'model_class': GRURegressor,
    'config': {
        'input_size': input_size,
        'hidden_size': 64,
        'num_layers': 1,
        'dropout': 0.0,
        'lr': 1e-3,
        'weight_decay': 0.0,   # NO L2
        'batch_size': 64,
        'epochs': base_epochs
    }
})

In [ ]:
# GRU configs (with L2)
experiments.append({
    'model_name': 'GRU_reg_configA',
    'model_class': GRURegressor,
    'config': {
        'input_size': input_size,
        'hidden_size': 128,
        'num_layers': 2,
        'dropout': 0.2,
        'lr': 1e-3,
        'weight_decay': 1e-5,
        'batch_size': 64,
        'epochs': base_epochs
    }
})

experiments.append({
    'model_name': 'GRU_reg_configB',
    'model_class': GRURegressor,
    'config': {
        'input_size': input_size,
        'hidden_size': 256,
        'num_layers': 3,
        'dropout': 0.3,
        'lr': 5e-4,
        'weight_decay': 1e-4,
        'batch_size': 64,
        'epochs': base_epochs
    }
})

In [ ]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# ---- Sequence creation ----
def create_sequences(data, targets, seq_length):
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i:i+seq_length])
        y.append(targets[i+seq_length])
    return np.array(X), np.array(y)

seq_length = 20
X_seq, y_seq = create_sequences(features_normalized, targets_normalized, seq_length)

# ---- Train/Test split (no shuffle for sequences) ----
X_train, X_test, y_train, y_test = train_test_split(
    X_seq, y_seq, test_size=0.33, shuffle=False
)

# ---- Convert to tensors ----
X_train_t = torch.tensor(X_train, dtype=torch.float32).to(device)
X_test_t  = torch.tensor(X_test, dtype=torch.float32).to(device)

y_train_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1).to(device)
y_test_t  = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1).to(device)

# ---- Models ----
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

class GRUModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers):
        super().__init__()
        self.gru = nn.GRU(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)
    def forward(self, x):
        out, _ = self.gru(x)
        return self.fc(out[:, -1, :])

# ---- Training function (FAST) ----
def train_model(model, lr, weight_decay, epochs=15):
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=64, shuffle=True)

    for epoch in range(epochs):
        model.train()
        for xb, yb in loader:
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
    return model

# ---- Evaluation ----
def evaluate(model):
    model.eval()
    with torch.no_grad():
        preds = model(X_test_t).cpu().numpy()
    preds_orig = target_scaler.inverse_transform(preds)
    y_orig = target_scaler.inverse_transform(y_test_t.cpu().numpy())
    return mean_squared_error(y_orig, preds_orig)

# ===== EXPERIMENTS =====
input_size = X_train_t.shape[2]
results = []

experiments = [
    ("LSTM_no_reg", LSTMModel(input_size, 64, 1), 1e-3, 0.0),
    ("GRU_no_reg",  GRUModel(input_size, 64, 1), 1e-3, 0.0),

    ("LSTM_reg_A",  LSTMModel(input_size, 128, 1), 1e-3, 1e-5),
    ("LSTM_reg_B",  LSTMModel(input_size, 64,  2), 5e-4, 1e-4),

    ("GRU_reg_A",   GRUModel(input_size, 128, 1), 1e-3, 1e-5),
    ("GRU_reg_B",   GRUModel(input_size, 64,  2), 5e-4, 1e-4),
]

start = time()
for name, model, lr, wd in experiments:
    print(f"\nTraining {name}...")
    model = model.to(device)
    train_model(model, lr, wd)
    mse = evaluate(model)
    results.append((name, mse))
    print(f"{name} — Test MSE: {mse:.4f}")

print(f"\nAll experiments completed in {(time()-start):.1f}s")

print("\n=== FINAL RESULTS ===")
for name, mse in results:
    print(f"{name:15s} | MSE: {mse:.4f}")


Device: cpu

Training LSTM_no_reg...
LSTM_no_reg — Test MSE: 6257666560.0000

Training GRU_no_reg...
GRU_no_reg — Test MSE: 5717413888.0000

Training LSTM_reg_A...
LSTM_reg_A — Test MSE: 6004643840.0000

Training LSTM_reg_B...
LSTM_reg_B — Test MSE: 6696586752.0000

Training GRU_reg_A...
GRU_reg_A — Test MSE: 6064652800.0000

Training GRU_reg_B...
GRU_reg_B — Test MSE: 6247015424.0000

All experiments completed in 372.0s

=== FINAL RESULTS ===
LSTM_no_reg     | MSE: 6257666560.0000
GRU_no_reg      | MSE: 5717413888.0000
LSTM_reg_A      | MSE: 6004643840.0000
LSTM_reg_B      | MSE: 6696586752.0000
GRU_reg_A       | MSE: 6064652800.0000
GRU_reg_B       | MSE: 6247015424.0000


Architecture signal: GRU (without regularization) outperformed LSTM_no_reg in these runs — suggests GRU may model your sequence dynamics better for this dataset/hyperparameter setup.

Regularization is mixed: “reg_A” helped the LSTM slightly (LSTM_reg_A < LSTM_no_reg), but “reg_B” clearly hurt LSTM. For GRU both reg variants performed worse than GRU_no_reg. This implies either:

regularization strength/type is not tuned (too strong in reg_B), or

regularization interacts with optimizer/learning rate / batch size in a way that harms training.

Effect sizes are modest: the RMSE differences (75.6k → 79.1k) are relatively small in absolute terms; whether they matter depends on the target scale and business tolerance.

Important caveat

You only reported one run per experiment. Single-run MSEs can be noisy. We cannot claim statistical significance from a single measurement — repeatability and variance are crucial.

Best performer: GRU_no_reg (lowest MSE & RMSE) — ~8.6% lower MSE than the LSTM_no_reg baseline.

Mixed effect of regularization: reg_A (moderate L2 + dropout) slightly helped both LSTM and GRU versus their unregularized counterparts in one case (LSTM_reg_A improved vs LSTM_no_reg), but LSTM_reg_B (larger model + stronger L2 + more dropout + smaller LR) hurt performance noticeably.

Model capacity & optimization interactions: larger hidden sizes / more layers (config B) combined with stronger weight decay and lower LR produced worse test error for LSTM (and only marginal change for GRU), suggesting either over-regularization or suboptimal optimization for the larger architectures.

**Note:** The California housing dataset was sourced by scikit-learn from the StatLib repository: https://www.dcc.fc.up.pt/~ltorgo/Regression/cal_housing.html

**Reference**  
Pace, R. K., & Barry, R. (1997). Sparse spatial autoregressions. Statistics & Probability Letters, 33(3), 291-297.